In [1]:
import os 
cwd = os.getcwd()
cwd

os.chdir("/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline")

new_cwd = os.getcwd() 

new_cwd


'/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline'

In [2]:
%run 00_inititial_transformations.ipynb

['2017-07-01', '2017-07-02', '2017-07-03', '2017-07-04', '2017-07-05'] ... ['2022-09-27', '2022-09-28', '2022-09-29', '2022-09-30', '2022-10-01']


In [3]:
from sklearn.metrics import r2_score

In [4]:
X_train.columns

Index(['branded_search_spend', 'nonbranded_search_spend', 'facebook_spend',
       'print_spend', 'ooh_spend', 'tv_spend', 'radio_spend', 'gdp',
       'base_rate', 'UNRATENSA_daily'],
      dtype='object')

In [5]:
#Defining the adstock transformation
def adstock(x, theta):
    return tsa.filters.recursive_filter(x, theta)

# Locking in final AdStock Scores
# This section should only apply adstock to the columns that need it.
X_train["branded_search_spend_adstock"] = adstock(X_train["branded_search_spend"], [0.9])
X_train["nonbranded_search_spend_adstock"] = adstock(X_train["nonbranded_search_spend"], [0.9])
X_train["facebook_spend_adstock"] = adstock(X_train["facebook_spend"], [0.9])



In [6]:
#Creating the lag variables

X_train['gdp_lag_1'] = X_train['gdp'].shift(-1)

X_train['base_rate_lag_1'] = X_train['base_rate'].shift(-1)

X_train['UNRATENSA_daily_lag_1'] = X_train['UNRATENSA_daily'].shift(-1)

X_train['sales_lag_1'] = y_train['sales'].shift(-1)

In [7]:
X_train.tail()

,branded_search_spend,nonbranded_search_spend,facebook_spend,print_spend,ooh_spend,tv_spend,radio_spend,gdp,base_rate,UNRATENSA_daily,branded_search_spend_adstock,nonbranded_search_spend_adstock,facebook_spend_adstock,gdp_lag_1,base_rate_lag_1,UNRATENSA_daily_lag_1,sales_lag_1
Week,,,,,,,,,,,,,,,,,
2021-06-13,1767.48,351.00,883.7400,0,0,0,0,5873553,0.08,6.1,21441.713106,3507.701367,10765.285101,5873553.0,0.08,6.1,72723.42
2021-06-20,1698.84,358.02,874.9026,0,0,0,0,5873553,0.08,6.1,20996.381796,3514.951230,10563.659191,5873553.0,0.08,6.1,72072.00
2021-06-27,1716.00,351.00,832.2600,0,0,0,0,5873553,0.08,6.1,20612.743616,3514.456107,10339.553272,5996233.0,0.10,5.7,73458.00
2021-07-04,1698.84,351.00,840.9258,0,0,0,0,5996233,0.10,5.7,20250.309255,3514.010496,10146.523744,5996233.0,0.10,5.7,72723.42
2021-07-11,1681.68,347.49,857.6568,0,0,0,0,5996233,0.10,5.7,19906.958329,3510.099447,9989.528170,NaN,NaN,NaN,NaN


In [8]:
y_train.columns

Index(['sales'], dtype='object')

In [9]:
#Adding only the required columns
X_train_final = X_train[[
    'sales_lag_1',
    "branded_search_spend_adstock",
    'nonbranded_search_spend_adstock',
    'facebook_spend_adstock',
    'print_spend',
    'gdp', 
    'base_rate', 
    'UNRATENSA_daily',
    'gdp_lag_1', 
    'base_rate_lag_1', 
    'UNRATENSA_daily_lag_1',
    ]]

In [10]:
#Starting the dfs from row 2, due to the lagging

X_train_final = X_train_final.iloc[:-1]

y_train_final = y_train.iloc[:-1]

In [11]:
#Cross-validation hold-out splits
hold_out_split = -26
y_train_calib = y_train_final.iloc[:hold_out_split]
y_train_hold_out = y_train_final.iloc[hold_out_split:]

X_train_calib = X_train_final.iloc[:hold_out_split]
X_val_hold_out = X_train_final.iloc[hold_out_split:]

In [12]:
type(X_train_calib)

pandas.core.frame.DataFrame

In [13]:
#Defining adj_R^2 function
def adj_r2(y_true, y_pred, p):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    #Making sure the denominator of adj R^2 > 0
    if n - p - 1 <= 0:
        return np.nan
    calculation = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return calculation


In [14]:
#Defining the function, whilst setting the Ridge alpha (lambda) to 10.
def test_model_adj_r2(X_train, X_val, y_train, y_val, regressors):
    ridge = Ridge(alpha=10, positive = True)
    
    #fitting the model
    ridge.fit(X_train, y_train)
    
    #predicting the x 
    y_pred = ridge.predict(X_val)
    
    score = adj_r2(y_val, y_pred, p=X_train.shape[1])
    print(f"{regressors}: Adjusted R² = {round(score, 4)}")    

#testing different numbers of regressors
X_1reg_train = X_train_calib.iloc[:, :1]
X_1reg_val = X_val_hold_out.iloc[:, :1]
test_model_adj_r2(X_1reg_train, X_1reg_val, y_train_calib, y_train_hold_out, "1 regressors")

X_2reg_train = X_train_calib.iloc[:, :2]
X_2reg_val = X_val_hold_out.iloc[:, :2]
test_model_adj_r2(X_2reg_train, X_2reg_val, y_train_calib, y_train_hold_out, "2 regressors")

X_3reg_train = X_train_calib.iloc[:, :3]
X_3reg_val = X_val_hold_out.iloc[:, :3]
test_model_adj_r2(X_3reg_train, X_3reg_val, y_train_calib, y_train_hold_out, "3 regressors")

X_4reg_train = X_train_calib.iloc[:, :4]
X_4reg_val = X_val_hold_out.iloc[:, :4]
test_model_adj_r2(X_4reg_train, X_4reg_val, y_train_calib, y_train_hold_out, "4 regressors")

X_5reg_train = X_train_calib.iloc[:, :5]
X_5reg_val = X_val_hold_out.iloc[:, :5]
test_model_adj_r2(X_5reg_train, X_5reg_val, y_train_calib, y_train_hold_out, "5 regressors")

X_6reg_train = X_train_calib.iloc[:, :6]
X_6reg_val = X_val_hold_out.iloc[:, :6]
test_model_adj_r2(X_6reg_train, X_6reg_val, y_train_calib, y_train_hold_out, "6 regressors")

X_7reg_train = X_train_calib.iloc[:, :7]
X_7reg_val = X_val_hold_out.iloc[:, :7]
test_model_adj_r2(X_7reg_train, X_7reg_val, y_train_calib, y_train_hold_out, "7 regressors")

X_8reg_train = X_train_calib.iloc[:, :8]
X_8reg_val = X_val_hold_out.iloc[:, :8] # Ensure this slice exists and has the same column
test_model_adj_r2(X_8reg_train, X_8reg_val, y_train_calib, y_train_hold_out, "8 regressors")

X_9reg_train = X_train_calib.iloc[:, :9]
X_9reg_val = X_val_hold_out.iloc[:, :9]
test_model_adj_r2(X_9reg_train, X_9reg_val, y_train_calib, y_train_hold_out, "9 regressors")

X_10reg_train = X_train_calib.iloc[:, :10]
X_10reg_val = X_val_hold_out.iloc[:, :10]
test_model_adj_r2(X_10reg_train, X_10reg_val, y_train_calib, y_train_hold_out, "10 regressors")

X_11reg_train = X_train_calib.iloc[:, :11]
X_11reg_val = X_val_hold_out.iloc[:, :11]
test_model_adj_r2(X_11reg_train, X_11reg_val, y_train_calib, y_train_hold_out, "11 regressors")

1 regressors: Adjusted R² = 0.7508
2 regressors: Adjusted R² = 0.7818
3 regressors: Adjusted R² = 0.7487
4 regressors: Adjusted R² = 0.7368
5 regressors: Adjusted R² = 0.7236
6 regressors: Adjusted R² = 0.71
7 regressors: Adjusted R² = 0.6939
8 regressors: Adjusted R² = 0.6759
9 regressors: Adjusted R² = 0.6684
10 regressors: Adjusted R² = 0.6463
11 regressors: Adjusted R² = 0.6211
